In [ ]:
import os

import psycopg2
import dotenv
import sqlalchemy as sa
from sqlalchemy.sql import text
import pandas as pd

dotenv.load_dotenv()

In [ ]:
engine = sa.create_engine(
    "{dialect}+{driver}://{username}:{password}@{host}:{port}/{database}".format(
        dialect="postgresql",
        driver="psycopg2",
        username=os.getenv('POSTGRES_USER'),
        password=os.getenv('POSTGRES_PASSWORD'),
        host=os.getenv('POSTGRES_HOST'),
        port=os.getenv("POSTGRES_PORT", '5432'),
        database=os.getenv('POSTGRES_DBNAME')
    ),
     connect_args={'sslmode':os.getenv("POSTGRES_SSL", 'disable')}
)


# SQL query to execute
query = "SELECT * FROM expense_detail;"

# Create connection and fetch data
with engine.connect() as db_conn:
    df = pd.read_sql_query(sql=text(query), con=db_conn)



In [ ]:
last_n_months = 6
df['yr_month'] = df['date'].dt.to_period('M').astype(str)

monthly_spends = []
expenses = []
for i, yr_month in enumerate(reversed(sorted(df['yr_month'].unique()))):
    if i >= last_n_months:
        break

    mth_df = df[df['yr_month'] == yr_month]

    monthly_spends.append(mth_df['amount'].sum())
    
    for ua in mth_df['expense_account'].unique():
        uadf = mth_df[ mth_df['expense_account'] == ua ]
        expenses.append( (yr_month, ua, uadf['amount'].sum()) )



In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('TkAgg')

sdf = pd.DataFrame(expenses, columns=['mth_yr', 'account', 'amount'])

df_pivot = sdf.pivot_table(index='mth_yr', columns='account', values='amount', aggfunc='sum')
df_pivot = df_pivot.fillna(0)


fig, ax = plt.subplots(figsize=(10, 6))
ax.stackplot(df_pivot.index, *df_pivot.T.values, labels=df_pivot.columns)

ax.set_ylim(0, max(monthly_spends))
fig.show()
fig.legend()